In [31]:
import networkx as nx
import matplotlib.pyplot as plt
import itertools
import more_itertools as mit
import numpy as np

In [63]:
# Needed for well-defined set inclusion later on
def set_maker(pi):
    return frozenset(frozenset(block) for block in pi)

In [99]:
def bond_checker(partition, G):

    """
    Checks whether a given partition of vertices of a graph G defines a bond
    """

    for block in partition:
        subgraph = nx.subgraph(G, block)
        if nx.is_connected(subgraph) == True:
            continue
        else:
            return False

    return True

In [100]:
def bond_sorter(bonds, n):  # try and work out if possible to do this without needing n input, purely working out number of nodes n from the bonds
    
    """
    Sorts bonds by rank order
    """
    ranks = []
    for pi in bonds:
        ranks.append(n - len(pi))

    return [x for _, x in sorted(zip(ranks, bonds))]

In [110]:
def bond_finder(G):

    """
    Finds all vertex partitions that define bonds of the graph G
    """

    # All partitions of the vertices
    partitions = mit.set_partitions(G.nodes)

    # Loop through all set partitions, check they are bonds, add to list of bonds if so
    bonds = []
    for p in partitions:
        verdict = bond_checker(p, G)
        if verdict == True:
            bonds.append(p) # turn blocks and partitions into sets and sort
    
    return bond_sorter(bonds, G.number_of_nodes())

In [116]:
def zeta_matrix(bonds):

    # Turn partitions and blocks within partitions into frozen sets so that set inclusion operations work
    bonds = [set_maker(p) for p in bonds]

    # Get number of bonds and initialise zeta matrix
    n = len(bonds)
    zeta = np.zeros((n, n))

    # Refinement check / filling in zeta matrix
    for i, pi in enumerate(bonds):
        for j, sigma in enumerate(bonds): # can possibly simplify further by working out how many bonds at each rank, then checking less bonds
            verdict = all(any(b <= B for B in sigma) for b in pi)
            if verdict == True:
                zeta[i, j] = 1

    return zeta

In [117]:
def full_inversion_mu(G):

    """
        For some graph G, finds all possible bonds, calculates the zeta and moebius matrices, and returns the require Moebius
        coefficients
    """

    bonds = bond_finder(G)
    zeta = zeta_matrix(bonds)
    mu_mat = np.linalg.inv(zeta)
    mu_values = mu_mat[0, :]

    return bonds, zeta, mu_mat, mu_values

In [127]:
bonds, zeta, mu_mat, mu_values = full_inversion_mu(nx.complete_graph(4))

In [129]:
print(zeta)

[[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
 [0. 1. 0. 0. 0. 0. 0. 1. 1. 0. 0. 1. 0. 0. 1.]
 [0. 0. 1. 0. 0. 0. 0. 1. 0. 1. 0. 0. 1. 0. 1.]
 [0. 0. 0. 1. 0. 0. 0. 1. 0. 0. 1. 0. 0. 1. 1.]
 [0. 0. 0. 0. 1. 0. 0. 0. 1. 1. 0. 0. 0. 1. 1.]
 [0. 0. 0. 0. 0. 1. 0. 0. 0. 1. 1. 1. 0. 0. 1.]
 [0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 1. 1. 1. 1.]
 [0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 1.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 1.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 1.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]]


In [130]:
print(mu_values)

[ 1. -1. -1. -1. -1. -1. -1.  2.  1.  2.  1.  2.  1.  2. -6.]
